# Quick Start

This page provides code examples for basic hyperalignment operations, starting with a two-subject, single-ROI Procrustes alignment. It then covers common-template construction, transformation matrix estimation, and applying the resulting alignment.

## 1. Two-Subject Example: Pairwise Procrustes Rotation

This example shows the basic two-subject Procrustes workflow. `sub1` is used as the source data matrix, `sub2` is used as the target data matrix, and `procrustes()` returns the T matrix that maps `sub1` into the target space. The aligned data are obtained with matrix multiplication: `sub1_aligned = sub1 @ T_sub1_to_sub2`.

In [ ]:
from pathlib import Path
import numpy as np

from fmriha.src.procrustes import procrustes
from fmriha.src.local_template import compute_template


# ---------------------------------------------------------------------
# Set paths and load two subjects from one ROI or local voxel set
# ---------------------------------------------------------------------
subject_dir = Path("/path/to/raw_data/subject_sl")

# Each array has shape: n_timepoints x n_vertices
sub1 = np.load(subject_dir / "sub1_sl.npy")
sub2 = np.load(subject_dir / "sub2_sl.npy")

print(f"Subject 1 data shape: {sub1.shape}")
print(f"Subject 2 data shape: {sub2.shape}")


# ---------------------------------------------------------------------
# Estimate and apply the Procrustes T matrix
# ---------------------------------------------------------------------
T_sub1_to_sub2 = procrustes(
    X=sub1,
    Y=sub2,
    reflection=True,
    scaling=False,
)

sub1_aligned_to_sub2 = sub1 @ T_sub1_to_sub2

print(f"T matrix shape: {T_sub1_to_sub2.shape}")
print(f"Aligned subject 1 shape: {sub1_aligned_to_sub2.shape}")

## 2. Multi-Subject Example: A Minimal Hyperalignment Workflow

The toolbox supports two common-template styles in this local example: a Procrustes-based template and a PCA-based template. In a real workflow, the Template, T Matrix, and Alignment steps should <span style="color: red; font-weight: bold;">NOT</span> all be estimated from exactly the same data. The example below uses one held-out group of subjects to construct the common template, then estimates T matrices and aligned data for a separate group of subjects with `subject_data @ subject_T`.

### Procrustes-Based Common Template

First, load the subjects reserved for template construction. Stack these matrices into `template_dss` with shape `n_subjects x n_timepoints x n_vertices`, then call `compute_template(..., kind="procrustes")`. These template subjects are not reused below for estimating T matrices or producing aligned data.

In [ ]:
# ---------------------------------------------------------------------
# Load subjects reserved for common-template construction
# ---------------------------------------------------------------------
sub3 = np.load(subject_dir / "sub3_sl.npy")

print(f"Subject 3 data shape: {sub3.shape}")

# Shape: n_subjects x n_timepoints x n_vertices
template_dss = np.stack([sub1, sub2, sub3], axis=0)
print(f"Template data stack shape: {template_dss.shape}")


# ---------------------------------------------------------------------
# Construct a Procrustes-based common template
# ---------------------------------------------------------------------
M_pr = compute_template(
    dss=template_dss,
    sl=None,
    kind="procrustes",
)

print(f"Procrustes template shape: {M_pr.shape}")

### PCA-Based Common Template

For a PCA-based common template, use `kind="pca"`. Setting `common_topography=True` returns the template in the original voxel or vertex space.

In [ ]:
# ---------------------------------------------------------------------
# Construct a PCA-based common template
# ---------------------------------------------------------------------
M_pca = compute_template(
    dss=template_dss,
    sl=None,
    kind="pca",
    common_topography=True,
)

print(f"PCA template shape: {M_pca.shape}")

### Transformation Matrix (T Matrix)

Estimate one T matrix per subject by aligning that subject's raw data to the selected common template. Here, the T matrices are estimated for a second group of subjects, separate from the subjects used to construct the template.

In [ ]:
# ---------------------------------------------------------------------
# Load subjects reserved for T-matrix estimation and alignment
# ---------------------------------------------------------------------
sub4 = np.load(subject_dir / "sub4_sl.npy")
sub5 = np.load(subject_dir / "sub5_sl.npy")
sub6 = np.load(subject_dir / "sub6_sl.npy")

print(f"Subject 4 data shape: {sub4.shape}")
print(f"Subject 5 data shape: {sub5.shape}")
print(f"Subject 6 data shape: {sub6.shape}")

# ---------------------------------------------------------------------
# Estimate subject-specific T matrices for the Procrustes-based common space
# ---------------------------------------------------------------------
sub4_pr_T = procrustes(X=sub4, Y=M_pr, reflection=True, scaling=False)
sub5_pr_T = procrustes(X=sub5, Y=M_pr, reflection=True, scaling=False)
sub6_pr_T = procrustes(X=sub6, Y=M_pr, reflection=True, scaling=False)

print(f"Subject 4 Procrustes T shape: {sub4_pr_T.shape}")


# ---------------------------------------------------------------------
# Estimate subject-specific T matrices for the PCA-based common space
# ---------------------------------------------------------------------
sub4_pca_T = procrustes(X=sub4, Y=M_pca, reflection=True, scaling=False)
sub5_pca_T = procrustes(X=sub5, Y=M_pca, reflection=True, scaling=False)
sub6_pca_T = procrustes(X=sub6, Y=M_pca, reflection=True, scaling=False)

print(f"Subject 4 PCA T shape: {sub4_pca_T.shape}")

### Alignment

Apply each subject-specific T matrix with standard matrix multiplication. The resulting matrices are expressed in the selected common space. This step uses the same second group of subjects used for T-matrix estimation, while the template still comes from the separate template group above.

In [ ]:
# ---------------------------------------------------------------------
# Align each subject to the Procrustes-based common space
# ---------------------------------------------------------------------
sub4_pr_aligned = sub4 @ sub4_pr_T
sub5_pr_aligned = sub5 @ sub5_pr_T
sub6_pr_aligned = sub6 @ sub6_pr_T

print(f"Subject 4 Procrustes-aligned shape: {sub4_pr_aligned.shape}")


# ---------------------------------------------------------------------
# Align each subject to the PCA-based common space
# ---------------------------------------------------------------------
sub4_pca_aligned = sub4 @ sub4_pca_T
sub5_pca_aligned = sub5 @ sub5_pca_T
sub6_pca_aligned = sub6 @ sub6_pca_T

print(f"Subject 4 PCA-aligned shape: {sub4_pca_aligned.shape}")